# Lab 3 — Probability for Pattern Recognition
**ENGS 3302 Pattern Recognition · Week 3** · Notes: *Week 3 — Probability for Pattern Recognition*

| Part | What you do | How |
|---|---|---|
| **1. Demo** | Long-run fractions, a table of counts, a discrete random variable, histogram to density, normal curves | All code is given. **Watch the lab video, run each cell, change values** |
| **2. Your turn** | Small tasks on the same ideas | Fill the `# TODO` cells. A check cell tells you if you are right |

This lab has **no grade**. The graded work this week is `Assignment_W03`.
In Colab: *File → Save a copy in Drive* first, then run the cells from top to bottom.

In [ ]:
# Run this cell first. It loads the libraries, a helper that checks your answers, and the penguin data.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

def check(label, got, expected, tol=1e-2):
    """Prints a green tick if `got` is close to `expected`, otherwise a red cross."""
    try:
        if isinstance(expected, str):
            ok = str(got) == expected
        else:
            ok = got is not None and np.allclose(np.asarray(got, dtype=float),
                                                 np.asarray(expected, dtype=float), atol=tol)
    except Exception:
        ok = False
    print(f"✅ {label}" if ok else f"❌ {label}: expected {expected}, got {got}")

# Palmer Penguins (Horst, Hill & Gorman 2020, CC0): 344 penguins of 3 species, measured in mm and g.
# Uses penguins.csv from the Moodle zip if it is next to the notebook, otherwise downloads it.
PENGUIN_FEATURES = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
def load_penguins():
    """Returns the cleaned table: 342 penguins with all four measurements."""
    try:
        df = pd.read_csv("penguins.csv")
    except FileNotFoundError:
        df = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv")
    return df.dropna(subset=PENGUIN_FEATURES).reset_index(drop=True)   # 2 penguins have no measurements

peng = load_penguins()
print("Ready.", len(peng), "penguins")

---
## Part 1 — Demo *(follow the lab video)*
### 1.1 Probability as a long-run fraction
Roll a fair die many times and count the sixes. The fraction gets close to 1/6 = 0.167 as the number of rolls grows.

In [ ]:
rng = np.random.default_rng(0)          # fixed seed: everyone gets the same "random" rolls
for n in (10, 100, 10000):
    rolls = rng.integers(1, 7, n)       # n whole numbers from 1 to 6
    print(f"{n:6d} rolls: fraction of sixes = {np.mean(rolls == 6):.4f}")

### 1.2 Probabilities from a table of counts
The day's catch from the Notes (Table 1): 100 fish, each with a class and *light* (lightness above 5.5) or *dark*.
We build the 100 rows and let pandas count.

In [ ]:
catch = pd.DataFrame({
    "species": ["sea bass"] * 60 + ["salmon"] * 40,
    "look":    ["light"] * 56 + ["dark"] * 4 + ["light"] * 3 + ["dark"] * 37})
table = pd.crosstab(catch["species"], catch["look"], margins=True)
print(table, "\n")

p_light_given_bass = 56 / 60           # inside the sea bass row
p_bass_given_light = 56 / 59           # inside the light column
print("P(light | sea bass) =", round(p_light_given_bass, 3))   # 0.933
print("P(sea bass | light) =", round(p_bass_given_light, 3))   # 0.949

# Bayes' rule gives the second number from the first
p_light = (56/60) * 0.6 + (3/40) * 0.4                       # total probability: 0.59
print("Bayes:              ", round((56/60) * 0.6 / p_light, 3))

### 1.3 A discrete random variable: sea bass in a net of three fish
Each fish is a sea bass with probability 0.6. We simulate 100 000 nets and compare with the exact PMF of the Notes (Worked example 3.4).

In [ ]:
rng = np.random.default_rng(2)
nets = (rng.random((100000, 3)) < 0.6).sum(axis=1)     # number of sea bass in each net
exact = {0: 0.064, 1: 0.288, 2: 0.432, 3: 0.216}
for k in range(4):
    print(f"X = {k}: simulated {np.mean(nets == k):.4f}   exact {exact[k]}")
print("average number of sea bass:", nets.mean(), "  expected value E[X] = 1.8")

### 1.4 From histogram to density
Flipper length of all 342 penguins, in 5 mm bins. Dividing each count by (total × bin width) gives a **density**:
now the area of each bar is the fraction of penguins in it, and all areas add up to 1.

In [ ]:
fl = peng["flipper_length_mm"].to_numpy()
edges = np.arange(170, 236, 5)
counts, _ = np.histogram(fl, edges)
density = counts / (len(fl) * 5)
print("counts:  ", counts)
print("bin 190-195: count", counts[4], " fraction", round(counts[4] / len(fl), 3), " height", round(density[4], 4))
print("total area =", np.sum(density * 5))

plt.hist(fl, bins=edges, density=True, color="lightgray", edgecolor="white")   # the same heights
plt.xlabel("flipper length (mm)"); plt.ylabel("density (per mm)"); plt.show()

### 1.5 Fitting a normal curve to each species
Mean and standard deviation of each species give a normal curve. `norm.pdf` gives its height, `norm.cdf` an area.

In [ ]:
xs = np.linspace(165, 240, 300)
colors = {"Adelie": "tab:orange", "Chinstrap": "tab:green", "Gentoo": "tab:blue"}
for sp, col in colors.items():
    v = peng.loc[peng["species"] == sp, "flipper_length_mm"]
    print(f"{sp:10s} n={len(v):3d}  mean={v.mean():.2f}  sd={v.std(ddof=0):.2f}")
    plt.hist(v, bins=edges, density=True, alpha=0.3, color=col)
    plt.plot(xs, norm.pdf(xs, v.mean(), v.std(ddof=0)), color=col, label=sp)
plt.legend(); plt.xlabel("flipper length (mm)"); plt.ylabel("density"); plt.show()

# Flipper lengths are whole mm, so the lengths 210 to 220 stand for the interval 209.5 to 220.5 mm.
g = peng.loc[peng["species"] == "Gentoo", "flipper_length_mm"]
model = norm.cdf(220.5, g.mean(), g.std(ddof=0)) - norm.cdf(209.5, g.mean(), g.std(ddof=0))
data = np.mean((g >= 210) & (g <= 220))
print(f"Gentoo, 210 to 220 mm:  normal model {model:.3f}   real fraction {data:.3f}")

### 1.6 A density can be larger than 1
The same Gentoo flippers in **metres**. The curve becomes 1000 times taller and 1000 times narrower; the area stays 1.

In [ ]:
m, s = g.mean(), g.std(ddof=0)
print("peak in mm:", round(norm.pdf(m, m, s), 4), " peak in m:", round(norm.pdf(m/1000, m/1000, s/1000), 2))
xm = np.linspace(0.18, 0.26, 2001)
print("area in m:", round(np.sum(norm.pdf(xm, m/1000, s/1000)) * (xm[1] - xm[0]), 3))

---
## Part 2 — Your turn
### Task 1 — Read the catch table
Use the table of 1.2. Compute $P(\text{dark})$, $P(\text{salmon}\mid\text{dark})$ and $P(\text{dark}\mid\text{salmon})$.

In [ ]:
p_dark = None               # TODO
p_salmon_given_dark = None  # TODO
p_dark_given_salmon = None  # TODO

In [ ]:
check("P(dark)", p_dark, 0.41)
check("P(salmon | dark)", p_salmon_given_dark, 0.902)
check("P(dark | salmon)", p_dark_given_salmon, 0.925)

### Task 2 — Bayes' rule as a function
Complete `bayes(p_b_given_a, p_a, p_b_given_not_a)`, which returns $P(A\mid B)$. Then use it for a catch with **20% sea bass**,
where $P(\text{light}\mid\text{sea bass})=56/60$ and $P(\text{light}\mid\text{salmon})=3/40$.

In [ ]:
def bayes(p_b_given_a, p_a, p_b_given_not_a):
    # TODO: total probability of B, then Bayes' rule
    return None

p20 = None   # TODO: P(sea bass | light) when 20% of the catch is sea bass

In [ ]:
check("bayes() on the catch table (60% sea bass)", bayes(56/60, 0.6, 3/40), 0.949)
check("P(sea bass | light), 20% sea bass", p20, 0.757)

### Task 3 — The PMF of a die
A fair die has $P(X=x)=1/6$ for $x=1,\dots,6$. Compute its expected value $E[X]=\sum x\,P(X=x)$ and $P(X\ge5)$ with NumPy.

In [ ]:
x = np.arange(1, 7)
pmf = np.full(6, 1/6)
expected = None   # TODO
p_ge5 = None      # TODO

In [ ]:
check("E[X] of a die", expected, 3.5)
check("P(X >= 5)", p_ge5, 0.333)

### Task 4 — Model and data for Chinstrap
Fit a normal curve to the **Chinstrap** flipper lengths (mean and `std(ddof=0)`). Compute the probability of the lengths 190 to 200 mm from the model
(as in 1.5, whole mm means the interval 189.5 to 200.5 mm), and the real fraction of Chinstrap samples with 190 to 200 mm.

In [ ]:
c = peng.loc[peng["species"] == "Chinstrap", "flipper_length_mm"]
p_model = None   # TODO: norm.cdf(...) - norm.cdf(...)
p_data = None    # TODO: fraction of samples with 190 <= c <= 200

In [ ]:
check("Chinstrap 190-200, normal model", p_model, 0.560, tol=0.002)
check("Chinstrap 190-200, real fraction", p_data, 0.559, tol=0.002)

### Task 5 — Which species is a flipper of 195 mm more typical of?
Compute the fitted densities $p(195\mid\text{Adelie})$ and $p(195\mid\text{Chinstrap})$, and store the name of the species with the higher one.

In [ ]:
p_adelie = None      # TODO
p_chinstrap = None   # TODO
more_typical = None  # TODO: "Adelie" or "Chinstrap"

In [ ]:
check("p(195 | Adelie)", p_adelie, 0.0454, tol=0.0005)
check("p(195 | Chinstrap)", p_chinstrap, 0.0560, tol=0.0005)
check("more typical of", more_typical, "Chinstrap")

### Reflect (2–3 sentences)
In 1.5 the normal model gives 0.579 for Gentoo flippers from 210 to 220 mm, but 0.602 of the real samples fall there. Why can a model and
the data disagree, and which one would you trust for new penguins? (Hint: what would happen to the model's number if you forgot that the lengths are whole mm?)

*Your answer:*